## 准备数据

In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [4]:
train_data, test_data = mnist_dataset()
print(train_data[0].shape, train_data[1].shape)
print(test_data[0].shape, test_data[1].shape)

(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [9]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal(shape=(28*28, 100)))
        self.b1 = tf.Variable(tf.zeros(shape=(100,)))
        self.W2 = tf.Variable(tf.random.normal(shape=(100, 10)))
        self.b2 = tf.Variable(tf.zeros(shape=(10,)))
        ####################
    
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 28*28])
        h = tf.matmul(x, self.W1) + self.b1
        h = tf.nn.relu(h)
        logits = tf.matmul(h, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()
optimizer = optimizers.Adam()

## 计算 loss

In [15]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [21]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 8.680727 ; accuracy 0.67185
epoch 1 : loss 8.66403 ; accuracy 0.6723167
epoch 2 : loss 8.647429 ; accuracy 0.6727
epoch 3 : loss 8.630919 ; accuracy 0.673
epoch 4 : loss 8.614499 ; accuracy 0.6735333
epoch 5 : loss 8.598171 ; accuracy 0.67395
epoch 6 : loss 8.581932 ; accuracy 0.67445
epoch 7 : loss 8.565778 ; accuracy 0.67483336
epoch 8 : loss 8.54971 ; accuracy 0.67508334
epoch 9 : loss 8.533731 ; accuracy 0.67553335
epoch 10 : loss 8.517838 ; accuracy 0.67605
epoch 11 : loss 8.502031 ; accuracy 0.67651665
epoch 12 : loss 8.486312 ; accuracy 0.6770333
epoch 13 : loss 8.470674 ; accuracy 0.67755
epoch 14 : loss 8.455122 ; accuracy 0.67795
epoch 15 : loss 8.439653 ; accuracy 0.67836666
epoch 16 : loss 8.424268 ; accuracy 0.67873335
epoch 17 : loss 8.408963 ; accuracy 0.67913336
epoch 18 : loss 8.393737 ; accuracy 0.6795167
epoch 19 : loss 8.3785925 ; accuracy 0.67981666
epoch 20 : loss 8.363528 ; accuracy 0.68023336
epoch 21 : loss 8.34854 ; accuracy 0.6806333
epoch 22 :